# Imports

In [1]:
from datasets import load_dataset
import pandas as pd
import random
import re
import json
import os
from dotenv import load_dotenv

load_dotenv()  # laedt GEMINI_API_KEY aus der .env in os.environ

True

# Daten anschauen

In [2]:
dataset = load_dataset("jfrei/GPTNERMED", trust_remote_code=True)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'jfrei/GPTNERMED' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Using the latest cached version of the dataset since jfrei/GPTNERMED couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/wielandcremer/.cache/huggingface/datasets/jfrei___gptnermed/default/1.0.0/a208175ae1717e6b84c8f7dcfdfd891d568d483a72cb393463de01c1e0cfaaa9 (last modified on Fri Jun 12 14:56:17 2026).


In [3]:
print(dataset)
print(dataset["train"].features)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['sentence', 'ner_labels'],
        num_rows: 7876
    })
    validation: Dataset({
        features: ['sentence', 'ner_labels'],
        num_rows: 984
    })
    test: Dataset({
        features: ['sentence', 'ner_labels'],
        num_rows: 985
    })
})
{'sentence': Value('string'), 'ner_labels': {'ner_class': List(ClassLabel(names=['Medikation', 'Dosis', 'Diagnose'])), 'start': List(Value('int32')), 'stop': List(Value('int32'))}}
{'sentence': '0,4 Diuretika 0,25 1x/die', 'ner_labels': {'ner_class': [1, 0, 1, 1], 'start': [0, 4, 14, 19], 'stop': [3, 13, 18, 25]}}


In [4]:
example = dataset["train"][0]
text = example["sentence"]
label = example["ner_labels"]
print(text)
print(label)

0,4 Diuretika 0,25 1x/die
{'ner_class': [1, 0, 1, 1], 'start': [0, 4, 14, 19], 'stop': [3, 13, 18, 25]}


In [5]:
for i in range(5):
    print(dataset["train"][i])

{'sentence': '0,4 Diuretika 0,25 1x/die', 'ner_labels': {'ner_class': [1, 0, 1, 1], 'start': [0, 4, 14, 19], 'stop': [3, 13, 18, 25]}}
{'sentence': '0,5mg Sildenafil bei erectilen Dysfunktion', 'ner_labels': {'ner_class': [1, 0, 2], 'start': [0, 6, 21], 'stop': [5, 16, 42]}}
{'sentence': '1.000 mg Phenprobenon über 24h, 200 mg Phenprobenon über 4 h.', 'ner_labels': {'ner_class': [1, 0, 1, 0], 'start': [0, 9, 32, 39], 'stop': [5, 21, 35, 51]}}
{'sentence': '10 Ranitidine', 'ner_labels': {'ner_class': [1, 0], 'start': [0, 3], 'stop': [2, 13]}}
{'sentence': '10 mg Enalapril 40mg Metoprolol 25mg', 'ner_labels': {'ner_class': [1, 0, 1, 0, 1], 'start': [0, 6, 16, 21, 32], 'stop': [5, 15, 20, 31, 36]}}


# LLM Generated Data (synthetic data)
- Paper probleme: Halluzinierte Begriffe und strukturelle Homogenität
- Idee: Echt medikationsnamen etc aus offiziellen Listen ziehen
- dazu: lables per matching erstellen, da LLMs teilweise schwierigkeiten damit haben

Hier: Versuche erstmal einen Satz zu erzeugung - genutzt dafür ein anteil von Beispiel Medikamenten und Diagnosen und Sätzen

In [2]:
FALLBACK_MEDS = ["Ramipril", "Metformin", "Ibuprofen", "Pantoprazol",
                 "Amoxicillin", "Bisoprolol", "Simvastatin", "L-Thyroxin"]
FALLBACK_DIAGS = ["arterielle Hypertonie", "Diabetes mellitus Typ 2", "Migraene",
                  "Refluxoesophagitis", "Pneumonie", "Hypothyreose",
                  "Vorhofflimmern", "Asthma bronchiale"]
DOSE_UNITS = ["mg", "ug", "g", "ml", "IE"]
SHAPES = [
    ("Medikation", "Dosis"),
    ("Medikation", "Dosis", "Diagnose"),
    ("Diagnose",),
    ("Medikation", "Diagnose"),
    ("Medikation", "Dosis", "Diagnose", "Diagnose"),
]

# original 12 sentences from prompt used to generate the GPTNERMED dataset
ORIGINAL_SENTENCES = [
    '<s>Zur weiteren Bekämpfung des <class="Diagnose">Juckreiz</class> wird die Einnahme von täglich <class="Dosis">100mg</class> <class="Medikation">Cortison</class> empfohlen.</s>',
    '<s>Bei wiederkehrender Infektion wie einer <class="Diagnose">Sepsis</class> oder schweren <class="Diagnose">Pnseumonien</class> wird eine Überwachung erforderlich sein.</s>',
    '<s><class="Medikation">Valsartan</class>/<class="Medikation">HCT</class> <class="Dosis">160</class>/<class="Dosis">12,5 mg</class> 1-0-0</s>',
    '<s><class="Medikation">Pantoprazol</class> <class="Dosis">40 mg</class> p.o.</s>',
    '<s>Die feingewebliche histopathologische Untersuchung ergab den Befund einer <class="Diagnose">Metastase</class> des bekannten malignen <class="Diagnose">Melanoms</class>.</s>',
    '<s><class="Diagnose">Diabetes Typ 2</class>-Patienten müssen regelmäßig <class="Medikation">Insulin</class> (mindestens mit <class="Dosis">12ml</class> dosiert) spritzen.</s>',
    '<s>Ich nehme <class="Medikation">Antibiotika</class> seit Tagen. Seitdem ist die <class="Diagnose">Mandelentzündung</class> deutlich besser geworden.</s>',
    '<s>Entlassung: <class="Dosis">40mg</class> <class="Medikation">Lidocain</class> wegen <class="Diagnose">Kopfschmerzen</class></s>',
    '<s>Zusammenfassende D: Zervix-PE bei 11 und 2 Uhr mit ausgeprägter <class="Diagnose">chronisch-florider Zervizitis</class>.</s>',
    '<s>Die Verschreibung von <class="Medikation">Hämatokrin</class> <class="Dosis">43mg</class> war unnötig.</s>',
    '<s>Der Patient klagt über <class="Diagnose">Karditiden</class> und nimmt täglich <class="Medikation">Nifedipin</class> ein.</s>',
    '<s>D: PE-Material der Portio bei 1 Uhr mit Nachweis einer schwergradigen <class="Diagnose">squamösen intraepithelialen Läsion</class> (<class="Diagnose">HSIL</class>; hier noch <class="Diagnose">CIN II</class>).</s>',
]

Beispiel Dosis

In [7]:
def make_dosage(rng):
    val = rng.choice([2.5, 5, 10, 12.5, 20, 25, 40, 50, 100, 250, 500])
    unit = rng.choice(DOSE_UNITS)
    
    space_prob = rng.random()
    if (space_prob < 0.7):
        text = f"{val}{unit}"
    else:
        text = f"{val} {unit}"

    return text.replace(".", ",")    

rng = random.Random(42)
print(make_dosage(rng))

500mg


Beispiel Entität: Medikation, Diagnose, Dosis

In [8]:
def sample_entities(rng, medication, diagnoses):
    entities = []
    for word in rng.choice(SHAPES):
        if word == "Medikation":
            entities.append({"text": rng.choice(medication), "label": "Medikation"})
        elif word == "Dosis":
            entities.append({"text": make_dosage(rng), "label": "Dosis"})
        elif word == "Diagnose":
            entities.append({"text": rng.choice(diagnoses), "label": "Diagnose"})
    
    return entities

rng = random.Random(42)
for _ in range(3):
    print(sample_entities(rng, FALLBACK_MEDS, FALLBACK_DIAGS))

[{'text': 'Ramipril', 'label': 'Medikation'}, {'text': '20ug', 'label': 'Dosis'}]
[{'text': 'Metformin', 'label': 'Medikation'}, {'text': '250ml', 'label': 'Dosis'}]
[{'text': 'Pantoprazol', 'label': 'Medikation'}, {'text': '12,5IE', 'label': 'Dosis'}]


## Gemini API
- test API aufruf

In [9]:
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="Antworte mit genau einem Satz: Was ist maschinelles Lernen?",
)

print(response.text)

ImportError: cannot import name 'genai' from 'google' (unknown location)

# Erste genierieung text

In [9]:
SYSTEM_PROMT = (
    "Du erzeugst einzelne synthetische deutsche medizinische Saetze fuer das "
    "Training eines NER-Modells. Regeln:\n"
    "1. Verwende JEDEN vorgegebenen Begriff WORTWOERTLICH und unveraendert, genau einmal.\n"
    "2. Fuege KEINE weiteren Medikamente, Dosierungen oder Diagnosen hinzu.\n"
    "3. Schreibe genau EINEN natuerlichen, plausiblen Satz im angegebenen Stil.\n"
    "4. Antworte ausschliesslich mit JSON, ohne Markdown und ohne Kommentar."
)

"""
{"text": "0,5mg Digoxin 0,25mg Lanatoside", "label": [[0, 5, "Dosis"], [6, 13, "Medikation"], [14, 20, "Dosis"], [21, 31, "Medikation"]]}
{"text": "0,5mg Sildenafil bei erectilen Dysfunktion", "label": [[0, 5, "Dosis"], [6, 16, "Medikation"], [21, 42, "Diagnose"]]}
"""
EXAMPLE_SAMPLES = [
    '{"text":"0,5mg Digoxin 0,25mg Lanatoside","entities":[{"text":"0,5mg","label":"Dosis"},{"text":"Digoxin","label":"Medikation"},{"text":"0,25mg","label":"Dosis"},{"text":"Lanatoside","label":"Medikation"}]}',
    '{"text":"0,5mg Sildenafil bei erectilen Dysfunktion","entities":[{"text":"0,5mg","label":"Dosis"},{"text":"Sildenafil","label":"Medikation"},{"text":"erectilen Dysfunktion","label":"Diagnose"}]}',

]

In [10]:
def build_task(entities):
    lines = ["Begriffe:"]
    for ent in entities:
        lines.append(f"- {ent['label']}: {ent['text']}")
    return "\n".join(lines)

print(build_task(sample_entities(rng, FALLBACK_MEDS, FALLBACK_DIAGS)))

Begriffe:
- Medikation: Pantoprazol
- Dosis: 500IE
- Diagnose: Asthma bronchiale
- Diagnose: Pneumonie


In [11]:
def build_format():
    lines = ["Format:"]
    lines.append('{"text": "...", "entities": [{"text": "...", "label": "Medikation|Dosis|Diagnose"}]}')
    return "\n".join(lines)

print(build_format())

Format:
{"text": "...", "entities": [{"text": "...", "label": "Medikation|Dosis|Diagnose"}]}


In [12]:
def build_llm_prompt(rng, medication, diagnoses, entities):

    message = []
    message.append(f"System: {SYSTEM_PROMT}")
    message.append("\n")
    message.append("\n")
    message.append("User:" + "\n")
    message.append(build_task(entities))
    message.append("\n")
    message.append("\n")
    message.append(build_format())
    message.append("\n")
    message.append("\n")
    message.append("Beispiel Sätze:\n")
    for sentence in EXAMPLE_SAMPLES:
        message.append(f"{sentence}\n")

    return "".join(message)

rng = random.Random(42)
entities = sample_entities(rng, FALLBACK_MEDS, FALLBACK_DIAGS)  
print(build_llm_prompt(rng, FALLBACK_MEDS, FALLBACK_DIAGS, entities))

System: Du erzeugst einzelne synthetische deutsche medizinische Saetze fuer das Training eines NER-Modells. Regeln:
1. Verwende JEDEN vorgegebenen Begriff WORTWOERTLICH und unveraendert, genau einmal.
2. Fuege KEINE weiteren Medikamente, Dosierungen oder Diagnosen hinzu.
3. Schreibe genau EINEN natuerlichen, plausiblen Satz im angegebenen Stil.
4. Antworte ausschliesslich mit JSON, ohne Markdown und ohne Kommentar.

User:
Begriffe:
- Medikation: Ramipril
- Dosis: 20ug

Format:
{"text": "...", "entities": [{"text": "...", "label": "Medikation|Dosis|Diagnose"}]}

Beispiel Sätze:
{"text":"0,5mg Digoxin 0,25mg Lanatoside","entities":[{"text":"0,5mg","label":"Dosis"},{"text":"Digoxin","label":"Medikation"},{"text":"0,25mg","label":"Dosis"},{"text":"Lanatoside","label":"Medikation"}]}
{"text":"0,5mg Sildenafil bei erectilen Dysfunktion","entities":[{"text":"0,5mg","label":"Dosis"},{"text":"Sildenafil","label":"Medikation"},{"text":"erectilen Dysfunktion","label":"Diagnose"}]}



In [13]:
from google import genai

def call_LLM(model, message):

    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

    response = client.models.generate_content(
        model=model,
        contents=message,
    )

    print(response.text)
    return response.text

model = "gemini-3.1-flash-lite"
message = build_llm_prompt(rng, FALLBACK_MEDS, FALLBACK_DIAGS, entities)
generated_sample = call_LLM(model, message)

ImportError: cannot import name 'genai' from 'google' (unknown location)

check

In [ ]:
print(f"LLM Prompt: \n {message}")
print(f"Entities: {entities}")
print("\n")
print(f"LLM Output: \n {generated_sample}")

LLM Prompt: 
 System: Du erzeugst einzelne synthetische deutsche medizinische Saetze fuer das Training eines NER-Modells. Regeln:
1. Verwende JEDEN vorgegebenen Begriff WORTWOERTLICH und unveraendert, genau einmal.
2. Fuege KEINE weiteren Medikamente, Dosierungen oder Diagnosen hinzu.
3. Schreibe genau EINEN natuerlichen, plausiblen Satz im angegebenen Stil.
4. Antworte ausschliesslich mit JSON, ohne Markdown und ohne Kommentar.

User:
Begriffe:
- Medikation: Ramipril
- Dosis: 20ug

Format:
{"text": "...", "entities": [{"text": "...", "label": "Medikation|Dosis|Diagnose"}]}

Beispiel Sätze:
{"text":"0,5mg Digoxin 0,25mg Lanatoside","entities":[{"text":"0,5mg","label":"Dosis"},{"text":"Digoxin","label":"Medikation"},{"text":"0,25mg","label":"Dosis"},{"text":"Lanatoside","label":"Medikation"}]}
{"text":"0,5mg Sildenafil bei erectilen Dysfunktion","entities":[{"text":"0,5mg","label":"Dosis"},{"text":"Sildenafil","label":"Medikation"},{"text":"erectilen Dysfunktion","label":"Diagnose"}]}



# Baue daraus Huggingface Datensatz form

In [14]:
#form anschauen
example = dataset["train"][0]
print(f"Form die ich habe: \n {generated_sample}")

print(f"Form die ich möchte: \n {example}")

NameError: name 'dataset' is not defined

In [15]:
def get_label_id(label):
    label_mapping = {
        "Medikation": 0,
        "Dosis": 1,
        "Diagnose": 2
    }
    return label_mapping.get(label)

def find_spans(sentence, entities):
    spans = []

    for ent in entities:
        ent_lower = ent["text"].lower()
        sentence = sentence.lower()
        
        start = sentence.find(ent_lower)
        stop = start + len(ent_lower)
        
        spans.append({
            "start": start,
            "stop": stop,
            "label": ent["label"]
        })

    spans.sort(key=lambda span: span["start"])

    return {
        "ner_class": [get_label_id(span["label"]) for span in spans],
        "start": [span["start"] for span in spans],
        "stop": [span["stop"] for span in spans]
    }

# Punkt 1: erst JSON parsen, dann die Begriffe im Satz-Text matchen
parsed = json.loads(generated_sample)
spans = find_spans(parsed["text"], entities)

print(parsed["text"])
print(entities)
print(spans)

NameError: name 'generated_sample' is not defined

In [ ]:
import json

def build_dataset_form(sentence, spans):
    sentence = json.loads(sentence)
    return {'sentence': sentence["text"], 'ner_labels': spans}

print(build_dataset_form(generated_sample, spans))

{'sentence': 'Der Patient nimmt täglich 20ug Ramipril ein.', 'ner_labels': {'ner_class': [1, 0], 'start': [26, 31], 'stop': [30, 39]}}


# Nun teste aus Medizinischen tabellen zu nehemn

In [3]:
import pandas as pd

AlphaIDSE = pd.read_csv(
    "/Users/wielandcremer/Dokumente/Uni/6_Semester/"
    "information_extraction_german_medical_data/"
    "meaningful_modification/data/medical_information/"
    "Alpha-ID-SE.csv",
    sep=";",
    encoding="utf-8-sig",
    dtype=str
)

whoACTddd = pd.read_csv("/Users/wielandcremer/Documents/Uni/6_Semester/information_extraction_german_medical_data/meaningful_modification/data/medical_information/who_atc_ddd.csv")

whoACTddd = whoACTddd.dropna(subset="ddd")
whoACTddd = whoACTddd.dropna(subset="uom")

In [4]:
MEDS = (whoACTddd["atc_name"].astype(str).to_list())
DOSE_UNITS = (whoACTddd["uom"].astype(str).to_list())
DOSE_VAL = (whoACTddd["ddd"].astype(str).to_list())
DIAGS = (AlphaIDSE["Diagnose"].astype(str).to_list())

In [5]:
SHAPES = [
    ("Medikation", "Dosis"),
    ("Medikation", "Dosis", "Diagnose"),
    ("Diagnose",),
    ("Medikation", "Diagnose"),
    ("Medikation", "Dosis", "Diagnose", "Diagnose"),
]

In [ ]:
print(MEDS)

['sodium fluoride', 'olaflur', 'hydrogen peroxide', 'chlorhexidine', 'amphotericin B', 'polynoxylin', 'domiphen', 'oxyquinoline', 'miconazole', 'natamycin', 'minocycline', 'magnesium hydroxide', 'algeldrate', 'dihydroxialumini sodium carbonate', 'calcium silicate', 'cimetidine', 'cimetidine', 'ranitidine', 'ranitidine', 'famotidine', 'famotidine', 'nizatidine', 'nizatidine', 'roxatidine', 'ranitidine bismuth citrate', 'lafutidine', 'misoprostol', 'enprostil', 'omeprazole', 'omeprazole', 'pantoprazole', 'pantoprazole', 'lansoprazole', 'rabeprazole', 'esomeprazole', 'esomeprazole', 'dexlansoprazole', 'carbenoxolone', 'sucralfate', 'pirenzepine', 'pirenzepine', 'bismuth subcitrate', 'proglumide', 'rebamipide', 'oxyphencyclimine', 'camylofin', 'camylofin', 'mebeverine', 'trimebutine', 'dicycloverine', 'dicycloverine', 'benzilone', 'glycopyrronium bromide', 'glycopyrronium bromide', 'glycopyrronium bromide', 'oxyphenonium', 'penthienate', 'propantheline', 'propantheline', 'methantheline', '

In [ ]:
print(DOSE_UNITS)

['mg', 'mg', 'mg', 'mg', 'mg', 'g', 'mg', 'mg', 'g', 'mg', 'mg', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'mg', 'mg', 'g', 'g', 'g', 'g', 'mg', 'mg', 'mcg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'g', 'g', 'g', 'mg', 'g', 'g', 'g', 'mg', 'g', 'g', 'g', 'g', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'g', 'g', 'mg', 'g', 'mg', 'g', 'g', 'mg', 'g', 'g', 'g', 'g', 'g', 'mg', 'mg', 'g', 'g', 'g', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'g', 'mg', 'mg', 'g', 'g', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'g', 'g', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'g', 'g', 'g', 'mg', 'g', 'g', 'mg', 'mg', 'mg', 'mg', 'g', 'g', 'mg', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'mg', 'g', 'MU', 'g', 'MU', 'g', 'g', 'g', 'g', 'MU', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'g', 'mg', 'g', 'mg', 'mg', 'g', 'mg', 'mg', 'g', 'g', 

## Code von oben für neuen fall angepasst 

In [6]:
def make_dosage(rng, values, units):
    val = rng.choice(values)
    unit = rng.choice(units)
    
    space_prob = rng.random()
    if (space_prob < 0.7):
        text = f"{val}{unit}"
    else:
        text = f"{val} {unit}"

    return text.replace(".", ",")    

rng = random.Random(42)
print(make_dosage(rng, DOSE_VAL, DOSE_UNITS))

1,4 mg


In [7]:
def sample_entities(rng, medication, diagnoses, dose_values, dose_units):
    entities = []
    for word in rng.choice(SHAPES):
        if word == "Medikation":
            entities.append({"text": rng.choice(medication), "label": "Medikation"})
        elif word == "Dosis":
            entities.append({"text": make_dosage(rng,dose_values, dose_units), "label": "Dosis"})
        elif word == "Diagnose":
            entities.append({"text": rng.choice(diagnoses), "label": "Diagnose"})
    
    return entities

rng = random.Random(42)
for _ in range(3):
    print(sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS))

[{'text': 'ondansetron', 'label': 'Medikation'}, {'text': '1,0mg', 'label': 'Dosis'}]
[{'text': 'hydroxychloroquine', 'label': 'Medikation'}, {'text': '7,5g', 'label': 'Dosis'}]
[{'text': 'tirofiban', 'label': 'Medikation'}, {'text': '10,0mg', 'label': 'Dosis'}]


In [8]:
"""
SYSTEM_PROMT = (
    "Du erzeugst einzelne synthetische deutsche medizinische Saetze fuer das "
    "Training eines NER-Modells. Regeln:\n"
    "1. Verwende JEDEN vorgegebenen Begriff WORTWOERTLICH und unveraendert, genau einmal.\n"
    "2. Fuege KEINE weiteren Medikamente, Dosierungen oder Diagnosen hinzu.\n"
    "3. Schreibe genau EINEN natuerlichen, plausiblen Satz im angegebenen Stil.\n"
    "4. Antworte ausschliesslich mit JSON, ohne Markdown und ohne Kommentar."
)
"""
SYSTEM_PROMT = (
    
    "Du erzeugst genau einen synthetischen deutschen medizinischen Satz für das Training eines NER-Modells.\n"

    "Verbindliche Regeln:\n"

    "1. Verwende jeden unter „Begriffe“ angegebenen Ausdruck exakt in der vorgegebenen Schreibweise, einschließlich Groß-/Kleinschreibung, Satzzeichen und Leerzeichen.\n"
    "2. Jeder vorgegebene Ausdruck muss im Feld „text“ genau einmal vorkommen.\n"
    "3. Verwende ausschließlich die vorgegebenen medizinischen Informationen.\n"
    "4. Ergänze keine weiteren Medikamente, Wirkstoffe, Dosierungen, Diagnosen, Symptome, Indikationen, Applikationswege, Darreichungsformen, Zeitangaben oder Behandlungseigenschaften.\n"
    "5. Allgemeine nichtmedizinische Funktionswörter und neutrale Satzbestandteile sind erlaubt.\n"
    "6. Erzeuge genau einen grammatisch vollständigen, plausiblen deutschen Satz.\n"
    "7. Erzeuge für jeden vorgegebenen Begriff genau einen Eintrag in „entities“.\n"
    "8. Der Wert von „entities.text“ muss exakt mit dem entsprechenden Ausdruck im Satz übereinstimmen.\n"
    "9. Die Einträge in „entities“ müssen in derselben Reihenfolge stehen, in der die Begriffe im Satz vorkommen.\n"
    "10. Verwende ausschließlich diese Labels: „Medikation“, „Dosis“ und „Diagnose“.\n"
    "11. Verwende keine Entität, die nicht in den Eingabebegriffen enthalten ist.\n"
    "12. Prüfe vor der Ausgabe intern:\n"
    "- Sind alle Begriffe exakt einmal enthalten?\n"
    "- Wurden keine medizinischen Informationen ergänzt?\n"
    "- Stimmen Text und Labels exakt überein?\n"
    "- Ist das JSON syntaktisch gültig?\n"
    "13. Antworte ausschließlich mit einem einzelnen gültigen JSON-Objekt. Kein Markdown, keine Erklärung und kein zusätzlicher Text.\n"
)

"""
{"text": "0,5mg Digoxin 0,25mg Lanatoside", "label": [[0, 5, "Dosis"], [6, 13, "Medikation"], [14, 20, "Dosis"], [21, 31, "Medikation"]]}
{"text": "0,5mg Sildenafil bei erectilen Dysfunktion", "label": [[0, 5, "Dosis"], [6, 16, "Medikation"], [21, 42, "Diagnose"]]}
"""
EXAMPLE_SAMPLES = [
    '{"text":"0,5mg Digoxin 0,25mg Lanatoside","entities":[{"text":"0,5mg","label":"Dosis"},{"text":"Digoxin","label":"Medikation"},{"text":"0,25mg","label":"Dosis"},{"text":"Lanatoside","label":"Medikation"}]}',
    '{"text":"0,5mg Sildenafil bei erectilen Dysfunktion","entities":[{"text":"0,5mg","label":"Dosis"},{"text":"Sildenafil","label":"Medikation"},{"text":"erectilen Dysfunktion","label":"Diagnose"}]}',

]

# original 12 sentences from prompt used to generate the GPTNERMED dataset
ORIGINAL_SENTENCES = [
    '<s>Zur weiteren Bekämpfung des <class="Diagnose">Juckreiz</class> wird die Einnahme von täglich <class="Dosis">100mg</class> <class="Medikation">Cortison</class> empfohlen.</s>',
    '<s>Bei wiederkehrender Infektion wie einer <class="Diagnose">Sepsis</class> oder schweren <class="Diagnose">Pnseumonien</class> wird eine Überwachung erforderlich sein.</s>',
    '<s><class="Medikation">Valsartan</class>/<class="Medikation">HCT</class> <class="Dosis">160</class>/<class="Dosis">12,5 mg</class> 1-0-0</s>',
    '<s><class="Medikation">Pantoprazol</class> <class="Dosis">40 mg</class> p.o.</s>',
    '<s>Die feingewebliche histopathologische Untersuchung ergab den Befund einer <class="Diagnose">Metastase</class> des bekannten malignen <class="Diagnose">Melanoms</class>.</s>',
    '<s><class="Diagnose">Diabetes Typ 2</class>-Patienten müssen regelmäßig <class="Medikation">Insulin</class> (mindestens mit <class="Dosis">12ml</class> dosiert) spritzen.</s>',
    '<s>Ich nehme <class="Medikation">Antibiotika</class> seit Tagen. Seitdem ist die <class="Diagnose">Mandelentzündung</class> deutlich besser geworden.</s>',
    '<s>Entlassung: <class="Dosis">40mg</class> <class="Medikation">Lidocain</class> wegen <class="Diagnose">Kopfschmerzen</class></s>',
    '<s>Zusammenfassende D: Zervix-PE bei 11 und 2 Uhr mit ausgeprägter <class="Diagnose">chronisch-florider Zervizitis</class>.</s>',
    '<s>Die Verschreibung von <class="Medikation">Hämatokrin</class> <class="Dosis">43mg</class> war unnötig.</s>',
    '<s>Der Patient klagt über <class="Diagnose">Karditiden</class> und nimmt täglich <class="Medikation">Nifedipin</class> ein.</s>',
    '<s>D: PE-Material der Portio bei 1 Uhr mit Nachweis einer schwergradigen <class="Diagnose">squamösen intraepithelialen Läsion</class> (<class="Diagnose">HSIL</class>; hier noch <class="Diagnose">CIN II</class>).</s>',
]

hier noch später testen, ob vielleicht mehr oder andere Sätze immer als Beispiel genutzt werden. Zum Beispiel könnte man hier zufällige sätze aus dem Datensatz nehmen und als Examples geben

In [43]:
CLASS_RE = re.compile(r'<class="([^"]+)">(.*?)</class>')

def parse_original_sentence(tagged):
    inner = tagged.replace("<s>", "").replace("</s>", "")

    entities = [
        {"text": match.group(2), "label": match.group(1)}
        for match in CLASS_RE.finditer(inner)
    ]
    text = CLASS_RE.sub(r"\2", inner)   # Tags entfernen, nur Inhalt behalten

    return {"text": text, "entities": entities}


EXAMPLE_POOL = [parse_original_sentence(sentence) for sentence in ORIGINAL_SENTENCES]


def build_examples(rng, pool=EXAMPLE_POOL, num_examples=2):
    chosen = rng.sample(pool, num_examples)
    lines = ["Beispiel Sätze:"]
    for example in chosen:
        lines.append(json.dumps(example, ensure_ascii=False))
    return "\n".join(lines)


# check
for example in EXAMPLE_POOL[:3]:
    print(json.dumps(example, ensure_ascii=False))

print("\n--- zwei random Beispiele ---")
print(build_examples(random.Random(42)))

{"text": "Zur weiteren Bekämpfung des Juckreiz wird die Einnahme von täglich 100mg Cortison empfohlen.", "entities": [{"text": "Juckreiz", "label": "Diagnose"}, {"text": "100mg", "label": "Dosis"}, {"text": "Cortison", "label": "Medikation"}]}
{"text": "Bei wiederkehrender Infektion wie einer Sepsis oder schweren Pnseumonien wird eine Überwachung erforderlich sein.", "entities": [{"text": "Sepsis", "label": "Diagnose"}, {"text": "Pnseumonien", "label": "Diagnose"}]}
{"text": "Valsartan/HCT 160/12,5 mg 1-0-0", "entities": [{"text": "Valsartan", "label": "Medikation"}, {"text": "HCT", "label": "Medikation"}, {"text": "160", "label": "Dosis"}, {"text": "12,5 mg", "label": "Dosis"}]}

--- zwei random Beispiele ---
Beispiel Sätze:
{"text": "Der Patient klagt über Karditiden und nimmt täglich Nifedipin ein.", "entities": [{"text": "Karditiden", "label": "Diagnose"}, {"text": "Nifedipin", "label": "Medikation"}]}
{"text": "Bei wiederkehrender Infektion wie einer Sepsis oder schweren Pnseumoni

In [44]:
def build_task(entities):
    lines = ["Begriffe:"]
    for ent in entities:
        lines.append(f"- {ent['label']}: {ent['text']}")
    return "\n".join(lines)


rng = random.Random(42)
entities = sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS)
task = build_task(entities)
print(task)

Begriffe:
- Medikation: c1-inhibitor, plasma derived
- Medikation: ondansetron
- Dosis: 1,0mg
- Diagnose: HIV-Krankheit mit progressivem asymptomatischen Kaposi-Sarkom


In [11]:
def build_format():
    lines = ["Format:"]
    lines.append('{"text": "...", "label": ["Medikation|Dosis|Diagnose"], ...}\n')
    return "\n".join(lines)

print(build_format())

Format:
{"text": "...", "label": ["Medikation|Dosis|Diagnose"], ...}



In [12]:
def build_llm_prompt(rng, medication, diagnoses, entities, example_pool=EXAMPLE_POOL, num_examples=2):

    message = []
    message.append(f"System: {SYSTEM_PROMT}")
    message.append("\n")
    message.append("\n")
    message.append("User:" + "\n")
    message.append(build_task(entities))
    message.append("\n")
    message.append("\n")
    message.append(build_format())
    message.append("\n")
    message.append("\n")
    message.append(build_examples(rng, example_pool, num_examples))
    message.append("\n")

    return "".join(message)

rng = random.Random(42)
entities = sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS)
print(build_llm_prompt(rng, FALLBACK_MEDS, FALLBACK_DIAGS, entities))

System: Du erzeugst genau einen synthetischen deutschen medizinischen Satz für das Training eines NER-Modells.
Verbindliche Regeln:
1. Verwende jeden unter „Begriffe“ angegebenen Ausdruck exakt in der vorgegebenen Schreibweise, einschließlich Groß-/Kleinschreibung, Satzzeichen und Leerzeichen.
2. Jeder vorgegebene Ausdruck muss im Feld „text“ genau einmal vorkommen.
3. Verwende ausschließlich die vorgegebenen medizinischen Informationen.
4. Ergänze keine weiteren Medikamente, Wirkstoffe, Dosierungen, Diagnosen, Symptome, Indikationen, Applikationswege, Darreichungsformen, Zeitangaben oder Behandlungseigenschaften.
5. Allgemeine nichtmedizinische Funktionswörter und neutrale Satzbestandteile sind erlaubt.
6. Erzeuge genau einen grammatisch vollständigen, plausiblen deutschen Satz.
7. Erzeuge für jeden vorgegebenen Begriff genau einen Eintrag in „entities“.
8. Der Wert von „entities.text“ muss exakt mit dem entsprechenden Ausdruck im Satz übereinstimmen.
9. Die Einträge in „entities“ müs

In [13]:
from google import genai

def call_LLM(model, message):
    
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

    response = client.models.generate_content(
        model=model,
        contents=message,
    )

    print(response.text)
    return response.text

model = "gemini-3.1-flash-lite"

for _ in range(3):
    entities = sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS)
    message = build_llm_prompt(rng, FALLBACK_MEDS, FALLBACK_DIAGS, entities)
    generated_sample = call_LLM(model, message)

{"text": "Patient mit HIV-Krankheit mit Pneumocystosis und Ektoper Aldosteron-produzierender Tumor des Hodens erhält warfarin 1,2mg.", "entities": [{"text": "HIV-Krankheit mit Pneumocystosis", "label": "Diagnose"}, {"text": "Ektoper Aldosteron-produzierender Tumor des Hodens", "label": "Diagnose"}, {"text": "warfarin", "label": "Medikation"}, {"text": "1,2mg", "label": "Dosis"}]}
{
  "text": "Der Patient leidet an Syndrom der Pulmonalklappenagenesie mit Fallot-Tetralogie und fehlendem Ductus arteriosus sowie Andersen-Tawil-Syndrom und erhält tropisetron 0,24 g.",
  "entities": [
    {
      "text": "Syndrom der Pulmonalklappenagenesie mit Fallot-Tetralogie und fehlendem Ductus arteriosus",
      "label": "Diagnose"
    },
    {
      "text": "Andersen-Tawil-Syndrom",
      "label": "Diagnose"
    },
    {
      "text": "tropisetron",
      "label": "Medikation"
    },
    {
      "text": "0,24 g",
      "label": "Dosis"
    }
  ]
}
{"text": "Bei dem Patienten mit Pädiatrisch akut auftr

In [14]:
def get_label_id(label):
    label_mapping = {
        "Medikation": 0,
        "Dosis": 1,
        "Diagnose": 2
    }
    return label_mapping.get(label)

def find_spans(sentence, entities):
    spans = []

    for ent in entities:
        ent_lower = ent["text"].lower()
        sentence = sentence.lower()
        
        start = sentence.find(ent_lower)
        stop = start + len(ent_lower)
        
        spans.append({
            "start": start,
            "stop": stop,
            "label": ent["label"]
        })

    spans.sort(key=lambda span: span["start"])

    return {
        "ner_class": [get_label_id(span["label"]) for span in spans],
        "start": [span["start"] for span in spans],
        "stop": [span["stop"] for span in spans]
    }


In [15]:

parsed = json.loads(generated_sample)
spans = find_spans(parsed["text"], entities)

print(parsed["text"])
print(entities)
print(spans)

Bei dem Patienten mit Pädiatrisch akut auftretendes neuropsychiatrisches Syndrom und Polyendokrines Polyneuropathie-Syndrom wird cefotiam in einer Menge von 0,8mg appliziert.
[{'text': 'cefotiam', 'label': 'Medikation'}, {'text': '0,8mg', 'label': 'Dosis'}, {'text': 'Pädiatrisch akut auftretendes neuropsychiatrisches Syndrom', 'label': 'Diagnose'}, {'text': 'Polyendokrines Polyneuropathie-Syndrom', 'label': 'Diagnose'}]
{'ner_class': [2, 2, 0, 1], 'start': [22, 85, 129, 157], 'stop': [80, 123, 137, 162]}


In [17]:
print(entities)
print(parsed["text"][22:80])
print(parsed["text"][85:123])
print(parsed["text"][129:137])

[{'text': 'cefotiam', 'label': 'Medikation'}, {'text': '0,8mg', 'label': 'Dosis'}, {'text': 'Pädiatrisch akut auftretendes neuropsychiatrisches Syndrom', 'label': 'Diagnose'}, {'text': 'Polyendokrines Polyneuropathie-Syndrom', 'label': 'Diagnose'}]
Pädiatrisch akut auftretendes neuropsychiatrisches Syndrom
Polyendokrines Polyneuropathie-Syndrom
cefotiam


In [18]:
import json

def build_dataset_form(sentence, spans):
    return {'sentence': sentence["text"], 'ner_labels': spans}


In [19]:
print(build_dataset_form(parsed, spans))

{'sentence': 'Bei dem Patienten mit Pädiatrisch akut auftretendes neuropsychiatrisches Syndrom und Polyendokrines Polyneuropathie-Syndrom wird cefotiam in einer Menge von 0,8mg appliziert.', 'ner_labels': {'ner_class': [2, 2, 0, 1], 'start': [22, 85, 129, 157], 'stop': [80, 123, 137, 162]}}


# Qwen lokal ausprobieren

In [ ]:
# ohne Systempromp für Qwen
def build_llm_prompt(rng, medication, diagnoses, entities, example_pool=EXAMPLE_POOL, num_examples=2):
    message = []
    message.append(build_task(entities))
    message.append("\n")
    message.append("\n")
    message.append(build_format())
    message.append("\n")
    message.append("\n")
    message.append(build_examples(rng, example_pool, num_examples))
    message.append("\n")

    return "".join(message)

rng = random.Random(42)
entities = sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS)
prompt = build_llm_prompt(rng, MEDS, DIAGS, entities)
print(prompt)

Begriffe:
- Medikation: ondansetron
- Dosis: 1,0mg

Format:
{"text": "...", "label": ["Medikation|Dosis|Diagnose"], ...}

Beispiel Sätze:
{"text": "D: PE-Material der Portio bei 1 Uhr mit Nachweis einer schwergradigen squamösen intraepithelialen Läsion (HSIL; hier noch CIN II).", "entities": [{"text": "squamösen intraepithelialen Läsion", "label": "Diagnose"}, {"text": "HSIL", "label": "Diagnose"}, {"text": "CIN II", "label": "Diagnose"}]}
{"text": "Bei wiederkehrender Infektion wie einer Sepsis oder schweren Pnseumonien wird eine Überwachung erforderlich sein.", "entities": [{"text": "Sepsis", "label": "Diagnose"}, {"text": "Pnseumonien", "label": "Diagnose"}]}



### unterteilen, um Modell nur einmal zu laden

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
#Qwen/Qwen2.5-3B-Instruct
#Qwen/Qwen2.5-0.5B-Instruct
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

In [ ]:
print("Lade Qwen-Modell ...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Lade Qwen-Modell ...


ValueError: The model's quantization config from the arguments has no `quant_method` attribute. Make sure that the model has been correctly quantized

In [ ]:
def call_qwen_LLM(message):
    messages = [
    {"role": "system", "content": SYSTEM_PROMT},
    {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    print(response)
    return response



In [ ]:
for _ in range(3):
    entities = sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS)
    message = build_llm_prompt(rng, MEDS, DIAGS, entities)
    generated_sample = call_qwen_LLM(message)

{"text": "Die Patientin wurde aufgrund einer akuten Gastroenteritis mit einer Medikation aus Ondansetron 1,0 mg täglichen Gebrauch empfohlen.", "entities": [{"text": "Ondansetron", "label": "Medikation"}, {"text": "1,0 mg", "label": "Dosis"}]}
{"text": "Die Patientin wird wegen einer Gastroektopathie auf ondansetron 1,0mg pro Tag empfohlen.", "entities": [{"text": "ondansetron", "label": "Medikation"}, {"text": "1,0mg", "label": "Dosis"}]}
{"text": "Die Patientin erhält eine tägliche Medikation mit 1,0mg ondansetron.", "label": ["Medikation|Dosis"]}


# MLX Ausprobieren damit schneller auf Mac

In [26]:
from mlx_lm import load, generate

MODEL_NAME = "mlx-community/Qwen2.5-3B-Instruct-4bit"
model, tokenizer = load(MODEL_NAME) 

def call_xml_qwen_LLM(message):
    messages = [
        {"role": "system", "content": SYSTEM_PROMT},
        {"role": "user",   "content": message},
    ]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    return generate(model, tokenizer, prompt=prompt, max_tokens=256, verbose=False)


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

In [ ]:
for _ in range(10):
    entities = sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS)
    message = build_llm_prompt(rng, MEDS, DIAGS, entities)
    generated_sample = call_xml_qwen_LLM(message)
    print(generated_sample)

{"text": "Bei wiederkehrender Infektion wie einer Sepsis oder schweren Pnseumonien wird eine Überwachung erforderlich sein.", "entities": [{"text": "Sepsis", "label": "Diagnose"}, {"text": "Pnseumonien", "label": "Diagnose"}]}
{"text": "Nicotine in einer Dosis von 1,5mg wird empfohlen, um die Erworbene hypothalamische Adipositas zu bessern.", "entities": [{"text": "Nicotine", "label": "Medikation"}, {"text": "1,5mg", "label": "Dosis"}, {"text": "Erworbene hypothalamische Adipositas", "label": "Diagnose"}]}
{"text": "Die Fatale postvirale neurodegenerative Krankheit deutet darauf hin, dass ich seit Monaten Itraconazole 0,2 mg pro Tag einnehmen muss.", "entities": [{"text": "Fatale postvirale neurodegenerative Krankheit", "label": "Diagnose"}, {"text": "Itraconazole", "label": "Medikation"}, {"text": "0,2 mg", "label": "Dosis"}]}
{"text": "testosterone seit Tagen. Die Symptome haben sich verbessert.", "entities": [{"text": "testosterone", "label": "Medikation"}]}
{"text": "guanoxan 10 mg

# Gemma ausprobieren

In [ ]:
from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler

#"mlx-community/gemma-2-9b-it-4bit"
MODEL_NAME = "mlx-community/gemma-2-2b-it-4bit"
# Alternativen:

model, tokenizer = load(MODEL_NAME)
sampler = make_sampler(temp=0.7, top_p=0.8)

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

In [31]:
def call_mlx_gemma_LLM(message):
    messages = [
        {"role": "user", "content": SYSTEM_PROMT + "\n\n" + message},
    ]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    return generate(model, tokenizer, prompt=prompt, max_tokens=256,
                    sampler=sampler, verbose=False)

In [32]:
for _ in range(10):
    entities = sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS)
    message = build_llm_prompt(rng, MEDS, DIAGS, entities)
    generated_sample = call_mlx_gemma_LLM(message)
    print(generated_sample)

```json
{"text": "Die Medikation Cefotiam in einer Dosis von 0,8mg soll bei dem Diagnose Polyendokrines Polyneuropathie-Syndrom eingesetzt werden.", "entities": [{"text": "Cefotiam", "label": "Medikation"}, {"text": "0,8mg", "label": "Dosis"}, {"text": "Polyendokrines Polyneuropathie-Syndrom", "label": "Diagnose"}]}
```
```json
{"text": "Die Diagnose HIV-Krankheit mit progressivem asymptomatischem Kaposi-Sarkom erfordert ständige Überwachung.", "entities": [{"text": "HIV-Krankheit", "label": "Diagnose"}, {"text": "Kaposi-Sarkom", "label": "Diagnose"}]}
```
```json
{"text": "Die letrozole-Medikation erfolgt mit einer Dosis von 0,15mg pro Tag.", "entities": [{"text": "letrozole", "label": "Medikation"}, {"text": "0,15mg", "label": "Dosis"}]}
```
```json
{"text": "Die Diagnose EPHB4-assoziierte generalisierte lymphatische Dysplasie mit nichtimmunologischem Hydrops fetalis erfordert die Dosierung von dobutamine in einer Dosis von 1,67mg.", "entities": [{"text": "EPHB4-assoziierte generalis

In [38]:
# ´´´json´´´wegbekommen
for _ in range(10):
    entities = sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS)
    message = build_llm_prompt(rng, MEDS, DIAGS, entities)
    generated_sample = call_mlx_gemma_LLM(message)
    generated_sample = re.sub(r"^```\w*\s*|\s*```$", "", generated_sample.strip())
    print(generated_sample)

{"text": "Die Medikation warfarin wird bei einer Dosis von 1,2 mg täglich verabreicht.", "entities": [{"text": "warfarin", "label": "Medikation"}, {"text": "1,2mg", "label": "Dosis"}]}
{"text": "Die Medikation Tropisetron in einer Dosis von 0,24 g wird zur Symptomatik-Steuerung eingesetzt.", "entities": [{"text": "Tropisetron", "label": "Medikation"}, {"text": "0,24 g", "label": "Dosis"}]}
{"text": "Für das Pädiatrisch akut auftretende neuropsychiatrische Syndrom wird cefotiam mit einer Dosis von 0,8 mg täglich verabreicht.", "entities": [{"text": "cefotiam", "label": "Medikation"}, {"text": "0,8 mg", "label": "Dosis"}]}
{"text": "Die Diagnose HIV-Krankheit mit progressiver asymptomatischer Kaposi-Sarkom erfordert eine sorgfältige Behandlung und Überwachung.", "entities": [{"text": "HIV-Krankheit", "label": "Diagnose"}, {"text": "Kaposi-Sarkom", "label": "Diagnose"}]}
{"text": "Die letrozole-Medikation wird bei einer Dosis von 0,15mg täglich eingenommen.", "entities": [{"text": "letroz

# Pipline verbesserungen, um mehr variablität in die Sätze zu bekommen
- behebt das Problem von GPTNERMED
- nutze Qwen ab hier wieder

In [20]:
STYLES = [
    "ein vollständiger, grammatischer Satz",
    "eine knappe, telegrammartige Klinik-Notiz (kein voller Satzbau)",
    "eine Medikationszeile mit Dosierschema (z.B. 1-0-0, p.o.)",
    "eine Aussage aus Patientensicht (Ich-Form)",
    "ein Auszug aus einem Arztbrief",
]
CONTEXTS = ["Aufnahme", "Anamnese", "Verlauf", "Entlassung", "Befund", "Medikationsplan"]


In [24]:
def random_shape(rng):
    n_med  = rng.choice([0, 1, 1, 2])
    n_diag = rng.choice([0, 1, 1, 2])
    n_dose = rng.choice([0, 1]) if n_med else 0
    shape = ["Medikation"]*n_med + ["Dosis"]*n_dose + ["Diagnose"]*n_diag
    shape = shape or ["Diagnose"]     # nie leer
    rng.shuffle(shape)
    return shape

SHAPES = [random_shape(rng) for _ in range(100)]


In [38]:
def build_llm_prompt(rng, medication, diagnoses, entities, example_pool=EXAMPLE_POOL, num_examples=2):

    message = []
    message.append(f"System: {SYSTEM_PROMT}")
    message.append("\n")
    message.append("\n")
    message.append("User:" + "\n")
    message.append(build_task(entities))
    message.append("\n")
    message.append("\n")
    message.append(f"Formuliere als: {rng.choice(STYLES)}. \nKontext: {rng.choice(CONTEXTS)}. ")
    message.append("\n")
    message.append("\n")
    message.append(build_format())
    message.append("\n")
    message.append("\n")
    message.append(build_examples(rng, example_pool, num_examples))
    message.append("\n")

    return "".join(message)

rng = random.Random(42)
entities = sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS)
print(build_llm_prompt(rng, FALLBACK_MEDS, FALLBACK_DIAGS, entities))

System: Du erzeugst genau einen synthetischen deutschen medizinischen Satz für das Training eines NER-Modells.
Verbindliche Regeln:
1. Verwende jeden unter „Begriffe“ angegebenen Ausdruck exakt in der vorgegebenen Schreibweise, einschließlich Groß-/Kleinschreibung, Satzzeichen und Leerzeichen.
2. Jeder vorgegebene Ausdruck muss im Feld „text“ genau einmal vorkommen.
3. Verwende ausschließlich die vorgegebenen medizinischen Informationen.
4. Ergänze keine weiteren Medikamente, Wirkstoffe, Dosierungen, Diagnosen, Symptome, Indikationen, Applikationswege, Darreichungsformen, Zeitangaben oder Behandlungseigenschaften.
5. Allgemeine nichtmedizinische Funktionswörter und neutrale Satzbestandteile sind erlaubt.
6. Erzeuge genau einen grammatisch vollständigen, plausiblen deutschen Satz.
7. Erzeuge für jeden vorgegebenen Begriff genau einen Eintrag in „entities“.
8. Der Wert von „entities.text“ muss exakt mit dem entsprechenden Ausdruck im Satz übereinstimmen.
9. Die Einträge in „entities“ müs

In [27]:
for _ in range(10):
    entities = sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS)
    message = build_llm_prompt(rng, MEDS, DIAGS, entities)
    generated_sample = call_xml_qwen_LLM(message)
    print(generated_sample)

{"text": "Für die Behandlung der Idiopathischen pleuropulmonaren Fibroelastose empfehle ich die Nebenstellung von Theophyllin 200 mg Täglich.", "entities": [{"text": "Theophyllin", "label": "Medikation"}, {"text": "Idiopathische pleuropulmonaren Fibroelastose", "label": "Diagnose"}]}
{"text": "Ich habe seit einigen Tagen Plattenepithelkarzinom der Leber und der intrahepatischen Gallengänge und nehme täglich ethosuximide ein.", "entities": [{"text": "Plattenepithelkarzinom der Leber und der intrahepatischen Gallengänge", "label": "Diagnose"}, {"text": "ethosuximide", "label": "Medikation"}]}
{"text": "levosulpiride wegen Lymphödem mit zerebraler arteriovenöser Fehlbildung und primärer pulmonaler Hypertonie", "label": ["Medikation|Diagnose"]}
{"text": "Methylmalonazidämie mit Homocystinurie-Patient muss homocystinurie (1x täglich) beheben.", "label": ["Medikation|Dosis|Diagnose"]}
{"text": "mofebutazone 0,2mg p.o.", "entities": [{"text": "mofebutazone", "label": "Medikation"}, {"text": "

## Nochmal gemini testen

In [23]:
from google import genai

def call_LLM(model, message):
    
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

    response = client.models.generate_content(
        model=model,
        contents=message,
    )

    print(response.text)
    return response.text

model = "gemini-3.1-flash-lite"

for _ in range(3):
    entities = sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS)
    message = build_llm_prompt(rng, FALLBACK_MEDS, FALLBACK_DIAGS, entities)
    generated_sample = call_LLM(model, message)

{"text": "Anamnese: mesna 1,2g.", "entities": [{"text": "mesna", "label": "Medikation"}, {"text": "1,2g", "label": "Dosis"}]}
{"text": "Ich nehme 0,2g pseudoephedrine ein.", "entities": [{"text": "0,2g", "label": "Dosis"}, {"text": "pseudoephedrine", "label": "Medikation"}]}
{
  "text": "Zur Behandlung der PLA2G6-assoziierter Dystonie-Parkinsonismus bei Aufnahme erfolgt die Gabe von dextromoramide in einer Dosis von 0,6g.",
  "entities": [
    {
      "text": "PLA2G6-assoziierter Dystonie-Parkinsonismus",
      "label": "Diagnose"
    },
    {
      "text": "dextromoramide",
      "label": "Medikation"
    },
    {
      "text": "0,6g",
      "label": "Dosis"
    }
  ]
}


# Die lokalen Modelle haben Problem mit den labels 
- Das build Format raus nehmen, fehler kommen beim bau der entities / labels, diese werden jedoch ohne hin nicht im json gebracht bei find spans
- Außerdem nur noch normale Sätze als Beispiel und nicht json

In [41]:
CLASS_RE = re.compile(r'<class="([^"]+)">(.*?)</class>')

def parse_original_sentence(tagged):
    inner = tagged.replace("<s>", "").replace("</s>", "")

    entities = [
        {"text": match.group(2), "label": match.group(1)}
        for match in CLASS_RE.finditer(inner)
    ]
    text = CLASS_RE.sub(r"\2", inner)   # Tags entfernen, nur Inhalt behalten

    return {"text": text, "entities": entities}


EXAMPLE_POOL = [parse_original_sentence(sentence) for sentence in ORIGINAL_SENTENCES]


def build_examples(rng, pool=EXAMPLE_POOL, num_examples=2):
    chosen = rng.sample(pool, num_examples)
    lines = ["Beispiel Sätze:"]
    for example in chosen:
        lines.append(json.dumps(example, ensure_ascii=False))
    return "\n".join(lines)


# check
for example in EXAMPLE_POOL[:3]:
    print(json.dumps(example, ensure_ascii=False))

print("\n--- zwei random Beispiele ---")
print(build_examples(random.Random(42)))

{"text": "Zur weiteren Bekämpfung des Juckreiz wird die Einnahme von täglich 100mg Cortison empfohlen.", "entities": [{"text": "Juckreiz", "label": "Diagnose"}, {"text": "100mg", "label": "Dosis"}, {"text": "Cortison", "label": "Medikation"}]}
{"text": "Bei wiederkehrender Infektion wie einer Sepsis oder schweren Pnseumonien wird eine Überwachung erforderlich sein.", "entities": [{"text": "Sepsis", "label": "Diagnose"}, {"text": "Pnseumonien", "label": "Diagnose"}]}
{"text": "Valsartan/HCT 160/12,5 mg 1-0-0", "entities": [{"text": "Valsartan", "label": "Medikation"}, {"text": "HCT", "label": "Medikation"}, {"text": "160", "label": "Dosis"}, {"text": "12,5 mg", "label": "Dosis"}]}

--- zwei random Beispiele ---
Beispiel Sätze:
{"text": "Der Patient klagt über Karditiden und nimmt täglich Nifedipin ein.", "entities": [{"text": "Karditiden", "label": "Diagnose"}, {"text": "Nifedipin", "label": "Medikation"}]}
{"text": "Bei wiederkehrender Infektion wie einer Sepsis oder schweren Pnseumoni

In [42]:
SYSTEM_PROMT = (
    
    "Du erzeugst genau einen synthetischen deutschen medizinischen Satz für das Training eines NER-Modells.\n"

    "Verbindliche Regeln:\n"

    "1. Verwende jeden unter „Begriffe“ angegebenen Ausdruck exakt in der vorgegebenen Schreibweise, einschließlich Groß-/Kleinschreibung, Satzzeichen und Leerzeichen.\n"
    "2. Jeder vorgegebene Ausdruck muss im Feld „text“ genau einmal vorkommen.\n"
    "3. Verwende ausschließlich die vorgegebenen medizinischen Informationen.\n"
    "4. Ergänze keine weiteren Medikamente, Wirkstoffe, Dosierungen, Diagnosen, Symptome, Indikationen, Applikationswege, Darreichungsformen, Zeitangaben oder Behandlungseigenschaften.\n"
    "5. Allgemeine nichtmedizinische Funktionswörter und neutrale Satzbestandteile sind erlaubt.\n"
    "6. Erzeuge genau einen grammatisch vollständigen, plausiblen deutschen Satz.\n"
    "7. Erzeuge für jeden vorgegebenen Begriff genau einen Eintrag in „entities“.\n"
    "8. Der Wert von „entities.text“ muss exakt mit dem entsprechenden Ausdruck im Satz übereinstimmen.\n"
    "9. Die Einträge in „entities“ müssen in derselben Reihenfolge stehen, in der die Begriffe im Satz vorkommen.\n"
    "10. Verwende ausschließlich diese Labels: „Medikation“, „Dosis“ und „Diagnose“.\n"
    "11. Verwende keine Entität, die nicht in den Eingabebegriffen enthalten ist.\n"
    "12. Prüfe vor der Ausgabe intern:\n"
    "- Sind alle Begriffe exakt einmal enthalten?\n"
    "- Wurden keine medizinischen Informationen ergänzt?\n"
    "- Stimmen Text und Labels exakt überein?\n"
    "- Ist das JSON syntaktisch gültig?\n"
    "13. Antworte ausschließlich mit einem einzelnen gültigen Satz, KEIN JSON. Kein Markdown, keine Erklärung und kein zusätzlicher Text.\n"
)

In [39]:
def build_llm_prompt(rng, medication, diagnoses, entities, example_pool=EXAMPLE_POOL, num_examples=2):

    message = []
    message.append(f"System: {SYSTEM_PROMT}")
    message.append("\n")
    message.append("\n")
    message.append("User:" + "\n")
    message.append(build_task(entities))
    message.append("\n")
    message.append("\n")
    message.append(f"Formuliere als: {rng.choice(STYLES)}. \nKontext: {rng.choice(CONTEXTS)}. ")
    message.append("\n")
    message.append("\n")
    message.append(build_examples(rng, example_pool, num_examples))
    message.append("\n")

    return "".join(message)

rng = random.Random(42)
entities = sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS)
print(build_llm_prompt(rng, FALLBACK_MEDS, FALLBACK_DIAGS, entities))

System: Du erzeugst genau einen synthetischen deutschen medizinischen Satz für das Training eines NER-Modells.
Verbindliche Regeln:
1. Verwende jeden unter „Begriffe“ angegebenen Ausdruck exakt in der vorgegebenen Schreibweise, einschließlich Groß-/Kleinschreibung, Satzzeichen und Leerzeichen.
2. Jeder vorgegebene Ausdruck muss im Feld „text“ genau einmal vorkommen.
3. Verwende ausschließlich die vorgegebenen medizinischen Informationen.
4. Ergänze keine weiteren Medikamente, Wirkstoffe, Dosierungen, Diagnosen, Symptome, Indikationen, Applikationswege, Darreichungsformen, Zeitangaben oder Behandlungseigenschaften.
5. Allgemeine nichtmedizinische Funktionswörter und neutrale Satzbestandteile sind erlaubt.
6. Erzeuge genau einen grammatisch vollständigen, plausiblen deutschen Satz.
7. Erzeuge für jeden vorgegebenen Begriff genau einen Eintrag in „entities“.
8. Der Wert von „entities.text“ muss exakt mit dem entsprechenden Ausdruck im Satz übereinstimmen.
9. Die Einträge in „entities“ müs

In [53]:
entities = sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS)
message = build_llm_prompt(rng, MEDS, DIAGS, entities)
generated_sample = call_xml_qwen_LLM(message)
print(generated_sample)

Pantoprazol 40 mg p.o.  
Buphenine 5 mg p.o.  
Dextromoramide 10 mg inj.  
Anamnese: Pädiatrisch akut auftretendes neuropsychiatrisches Syndrom


In [54]:
spans = find_spans(generated_sample, entities)
print(generated_sample)
print(entities)
print(spans)

Pantoprazol 40 mg p.o.  
Buphenine 5 mg p.o.  
Dextromoramide 10 mg inj.  
Anamnese: Pädiatrisch akut auftretendes neuropsychiatrisches Syndrom
[{'text': 'buphenine', 'label': 'Medikation'}, {'text': 'dextromoramide', 'label': 'Medikation'}, {'text': 'Pädiatrisch akut auftretendes neuropsychiatrisches Syndrom', 'label': 'Diagnose'}]
{'ner_class': [0, 0, 2], 'start': [25, 47, 85], 'stop': [34, 61, 143]}


In [40]:
for _ in range(10):
    entities = sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS)
    message = build_llm_prompt(rng, MEDS, DIAGS, entities)
    generated_sample = call_xml_qwen_LLM(message)
    print(generated_sample)

Der Patient wurde bei der Anamnese mit Pantoprazol 40 mg p.o. behandelt und zeigte eine ausgeprägte chronisch-florider Zervizitis bei 11 und 2 Uhr an der Cervix-PE.
Die Verschreibung von chloroquine 0,24g und oxycodone war unnötig. Der Septische Schock durch Escherichia coli ist beseitigt.
Patient wird für Plattenepithelkarzinom der Leber und intrahepatischen Gallengänge 40,0g estriol p.o. empfohlen.
Am Abend wurde der Patient aufgenommen. Seitdem nimmt er seit Tagen budesonide 0,15mg pro Tag. Diagnose: Hernia diaphragmatica mit Einklemmung und chronischer ulzeröser Cameron-Läsion, ohne Blutung.
Bei einer EPHB4-assoziierten generalisierten lymphatische Dysplasie mit nichtimmunologischem Hydrops fetalis wird eine Überwachung erforderlich sein. Sulfamethoxazole 1-0-0, acetazolamide 25 mg p.o. p.o. eingesetzt.
Bei Aufnahme wurde ein Pantoprazol 40 mg p.o. eingenommen, da das Yuan-Harel-Lupski-Syndrom die Hereditäre Persistenz des fetalen Hämoglobins mit Sichelzellkrankheit bei wiederkehre

In [ ]:
for _ in range(10):
    entities = sample_entities(rng, MEDS, DIAGS, DOSE_VAL, DOSE_UNITS)
    message = build_llm_prompt(rng, MEDS, DIAGS, entities)
    generated_sample = call_xml_qwen_LLM(message)

    parsed = json.loads(generated_sample)
    spans = find_spans(parsed["text"], entities)

    print(f"Generated Sample:\n {generated_sample}")
    
    